# 6.3 标签偏移

## 双样本检验 (Two-Sample Test)
假设训练集样本来自于分布 $P$，测试集样本来自于分布 $Q$。

判断两个数据集（比如训练集和测试集）是否来自于同一个概率分布：

- 基于一个二分类器
- 最大均值差异 (Maximum Mean Discrepancy, MMD)：通过比较两个分布在特征空间中的“平均位置”来判断差异。
    - 均值嵌入 (Mean Embedding)：MMD 将分布 $P$ 映射到再生核希尔伯特空间 (RKHS) 中的一个点（均值向量）$$\mu_P = E_{x \sim P} [\phi(x)]$$
    这里的 $\phi(x)$ 是由核函数 $k(x, x')$ 隐含定义的特征映射。
    - MMD 距离定义：两个均值嵌入点之间的欧氏距离：
    $$MMD(P, Q) = \| \mu_P - \mu_Q \|_{\mathcal{H}}$$
    - 平方 MMD 公式（实际计算中使用的公式）：不需要显式计算 $\phi(x)$，而是利用核函数 $k$：$$MMD^2(P, Q) = E_{x,x' \sim P}[k(x,x')] + E_{y,y' \sim Q}[k(y,y')] - 2E_{x \sim P, y \sim Q}[k(x,y)]$$
    训练集内部的相似度 + 测试集内部的相似度 + 训练集与测试集之间的相似度
- 柯尔莫哥洛夫-斯米尔诺夫检验 (K-S Test)：通常用于一维数据，比较累积分布函数的最大差值。
    - 经验累积分布函数 (ECDF)：对于样本量为 $n$ 的数据集 $X$，定义 $$F_n(x) = \frac{1}{n} \sum_{i=1}^n I_{(-\infty, x]}(x_i)$$
    $I$ 是指示函数，表示小于或等于 $x$ 的样本所占的比例。
    - 统计量 $D$：K-S 检验计算两个 ECDF 之间的最大绝对差值：$D_{n,m} = \sup_x |F_n(x) - G_m(x)|$，其中 $F_n$ 是训练集分布，$G_m$ 是测试集分布。
    - 判定标准：当样本量足够大时，如果 $D_{n,m} > \text{threshold}$，有足够的证据拒绝“分布一致”的假设。这个阈值通常由显著性水平（如 $\alpha=0.05$）决定。
- Kullback-Leibler 散度 (KL divergence)：用分布 $Q$ 来近似真实分布 $P$，这个过程中损失了多少信息
$$D_{KL}(P || Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$$

### 标签偏移

标签偏移 (Label Shift)：标签的边缘分布 $p(y)$ 发生变化 -> $q(y)$，但给定标签后的特征条件分布 $p(x|y)$ 保持不变，更符合“环境变化，但物理规律不变”的直觉（因果模型）。

给定一个结果，可能导致该结果的原因是不变的

- 训练场景：在平常时期训练一个感冒诊断模型，此时患流感的人比例很低 ($p(y)$ 较小)。

- 测试场景：到了流感爆发季节，患病比例大幅上升 ($q(y) \ne p(y)$)。

- 关键假设：虽然生病的人变多了，但感冒引起的症状（如发烧、咳嗽等特征 $x$）在患病人群中的表现 $p(x|y)$ 是稳定的 。

### 修正方法
**知道测试集分布 q(y)时**
- 给定训练好的模型 $p(y \mid x)$
- We want $q(y \mid x)$ 

$$\underbrace{q(y \mid x) = \frac{q(x \mid y)\,q(y)}{q(x)}}_{\text{Bayes Rule}} = \frac{p(x \mid y)\,p(y)}{p(x)} \cdot \frac{q(y)}{p(y)} \cdot \underbrace{\frac{p(x)}{q(x)}}_{\text{Drop this}} \;\propto\; p(y \mid x)\,\frac{q(y)}{p(y)}$$

- 给训练集中的每个样本 $(x_i, y_i)$ 分配一个权重系数 $\beta(y_i)$：$$\beta(y) = \frac{q(y)}{p(y)}$$
- 修正后的目标函数：$$\text{minimize}_f \frac{1}{m} \sum_{i=1}^{m} \beta(y_i) \cdot l(f(x_i), y_i)$$

**黑盒偏移估计 (BBSE)**

前提假设：这个黑盒模型必须足够好，使得混淆矩阵 $C$ 是 **满秩（可逆）** 的
1. 构建混淆矩阵 (Confusion Matrix, $C$)：利用**有标签的验证集**，看看模型在各个类别上“犯错”的概率。
2. 计算测试集的平均预测值 ($\hat{\mu}$)：把模型在测试集上跑一遍，算出模型预测“猫”的平均概率是多少。
3. 解线性方程组：利用数学公式 $C \cdot w = \hat{\mu}$ 解出权重 $w$。
- BBSE 算法的计算复杂度随类别数量 $k$ 呈立方级增长 ($O(k^3)$)


### 复杂场景挑战

**流式数据 (Streaming Data)**
- 在实际生产环境中，数据往往不是一次性给齐的测试集，而是源源不断流进来的
- 需要在观察数据的同时估计权重 $\beta(y)$
- 利用 SGD 迭代调整权重，让当前的预测分布逐渐靠拢训练集的加权分布

**大规模标签集 (Large Label Sets)**

BBSE $O(k^3)$ 计算开销将无法承受

- 特征矩匹配 (via MMD)：不再对比具体的类别输出，而是对比高维特征空间中的统计特性（矩）。
- GAN 矩匹配：利用对抗生成网络的思想，让一个判别器去学习区分训练集和测试集的分布差异 。
- 分值分类器 (Classifier of scores)：训练一个分类器来给训练集和测试集的样本打分，从而直接估计概率比 。

**更优的优化目标 (Better Objective)**

- 为了让模型在新环境下的表现更加精准，我们需要更好的**校准（Calibration）**手段。

- KL 散度 (Kullback-Leibler Divergence)：使用 KL 散度作为损失函数，来校准我们估计的 $q(y)$ 与加权后的 $p(y)$ 之间的差异 。这能确保我们修正后的分布在概率逻辑上是最接近真实情况的。

### 非平稳性与动态变化 (Non-Stationarity)